In [18]:
%load_ext autoreload
%autoreload 2

In [19]:
import pandas as pd
import numpy as np
from pyMyriad import *
from pyMyriad.tabular import flatten, tabulate
from pyMyriad.listing import gt_table, simple_table, cascade_table

In [20]:
df = pd.DataFrame({
  "id": np.arange(1000),
  "Gender": np.random.choice(["M", "F"], 1000),
  "Age": np.random.randint(18, 70, 1000),
  "Income": np.random.normal(50000, 15000, 1000),
  "Country": np.random.choice(["Benin", "Albania"], 1000),
  "Education": np.random.normal(0, 1, 1000),
  "Employed": np.random.choice([0, 1], 1000)
})

In [21]:
mfun = lambda df: np.mean(df.Income)
nfun = lambda df: np.std(df.Income)
efun = lambda df: np.mean(df.Education)
benin_fun =  lambda df: df.Country == 'Benin'
age_40 = lambda df: df.Age > 40
age_60 = lambda df: df.Age > 60

atree = AnalysisTree()\
  .split_by("df.Gender")\
  .split_by(label = "Benin Y/N", expr = benin_fun)\
  .split_by(label = "Age Group", age40 = age_40, age60 = age_60)\
  .analyze_by(m = mfun, n = nfun, label = "Income analysis")\
  .analyze_by(m = efun, label = "Education analysis")

res = atree.run(df)


# Tabulation with simple_table()

The `simple_table()` function provides a lightweight way to display analysis results as a pandas DataFrame without requiring the great-tables package.

In [22]:
# Basic simple_table - shows only analysis results
simple_table(res)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
7,Gender,F,Benin Y/N,False,Age Group,age40,m,49412.408323
7,,,,,,,n,15025.433302
8,,,,,,,m,0.07254
10,,,,,,age60,m,47662.31598
10,,,,,,,n,17303.053207
11,,,,,,,m,0.080393
15,,,,True,,age40,m,50360.355137
15,,,,,,,n,15114.99013
16,,,,,,,m,-0.119434
18,,,,,,age60,m,50358.625723


## simple_table with pivot

You can pivot by a split variable to show results across columns:

In [23]:
# Pivot by Gender to show results side-by-side
simple_table(res, by="df.Gender")

pivot_lvl,_Level_0,_Level_1,_Level_2,_Level_3,Statistic,F,M
0,Benin Y/N,False,Age Group,age40,m,0.07254,-0.131248
1,,,,,m,49412.408323,48211.790437
2,,,,,n,15025.433302,15381.549225
3,,,,age60,m,0.080393,-0.071286
4,,,,,m,47662.31598,48229.692543
5,,,,,n,17303.053207,14762.900535
6,,True,,age40,m,-0.119434,0.051745
7,,,,,m,50360.355137,49183.822877
8,,,,,n,15114.99013,14423.260704
9,,,,age60,m,-0.170524,0.171555


## cascade_table - show all tree nodes

Include all tree nodes (splits, summaries, and analyses) using `cascade_table()`:

In [24]:
# Show all nodes including splits
cascade_table(res).head(10)

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
0,--,--,--,--,--,--,None,None
1,Gender,--,--,--,--,--,None,None
2,,F,--,--,--,--,None,None
3,,,Benin Y/N,--,--,--,None,None
4,,,,False,--,--,None,None
5,,,,,Age Group,--,None,None
6,,,,,,age40,None,None
7,,,,,,,m,49412.408323
7,,,,,,,n,15025.433302
8,,,,,,,m,0.07254


## simple_table without duplicate suppression

By default, duplicate values in hierarchy columns are suppressed for cleaner display. You can disable this:

In [25]:
# Show all duplicate values
simple_table(res, suppress_duplicates=False).head()

,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,Statistic,Value
7,Gender,F,Benin Y/N,False,Age Group,age40,m,49412.408323
7,Gender,F,Benin Y/N,False,Age Group,age40,n,15025.433302
8,Gender,F,Benin Y/N,False,Age Group,age40,m,0.07254
10,Gender,F,Benin Y/N,False,Age Group,age60,m,47662.31598
10,Gender,F,Benin Y/N,False,Age Group,age60,n,17303.053207


# Tabulation with gt_table()

The `gt_table()` function creates beautifully formatted tables using the great-tables package. It provides more styling options and is ideal for reports and presentations.

In [26]:
# Basic gt_table with title
gt_table(res, title="Analysis Results").show()

Analysis Results 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49412.40832296735 
 
 
 
 
 
 
 
 
 n 
 15025.433301699664 
 
 
 
 
 
 
 
 
 m 
 0.07254044114026609 
 
 
 
 
 
 
 
 age60 
 m 
 47662.31598042284 
 
 
 
 
 
 
 
 
 n 
 17303.053206656172 
 
 
 
 
 
 
 
 
 m 
 0.08039270194225605 
 
 
 
 
 
 True 
 
 age40 
 m 
 50360.35513714851 
 
 
 
 
 
 
 
 
 n 
 15114.99013027268 
 
 
 
 
 
 
 
 
 m 
 -0.11943426225447411 
 
 
 
 
 
 
 
 age60 
 m 
 50358.62572299295 
 
 
 
 
 
 
 
 
 n 
 15356.269397920525 
 
 
 
 
 
 
 
 
 m 
 -0.17052401128244693 
 
 
 
 M 
 
 False 
 
 age40 
 m 
 48211.790436672985 
 
 
 
 
 
 
 
 
 n 
 15381.549224855946 
 
 
 
 
 
 
 
 
 m 
 -0.13124823552832085 
 
 
 
 
 
 
 
 age60 
 m 
 48229.69254328166 
 
 
 
 
 
 
 
 
 n 
 14762.900535368122 
 
 
 
 
 
 
 
 
 m 
 -0.0712858978586298 
 
 
 
 
 
 True 
 
 age40 
 m 
 49183.82287650288 
 
 
 
 
 
 
 
 
 n 
 14423.26070433845 
 
 
 
 
 
 
 
 
 m 
 0.05174539786330165 
 
 
 
 
 
 
 
 age60 
 m 
 50243.500038446844 
 
 
 
 
 
 
 
 
 n 
 14194.466442890502 
 
 
 
 
 
 
 
 
 m 
 0.1715553417547939

## gt_table with pivot

Pivot by a split variable:

In [27]:
# Pivot by Gender with custom title and subtitle
gt_table(
    res, 
    by="df.Gender", 
    title="Gender Comparison",
    subtitle="Income and Education Analysis by Gender"
).show()

Gender Comparison 
 
 
 Income and Education Analysis by Gender 
 
 
 
 Hierarchy 
 
 Statistic 
 F 
 M 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 0.07254044114026609 
 -0.13124823552832085 
 
 
 
 
 
 
 m 
 49412.40832296735 
 48211.790436672985 
 
 
 
 
 
 
 n 
 15025.433301699664 
 15381.549224855946 
 
 
 
 
 
 age60 
 m 
 0.08039270194225605 
 -0.0712858978586298 
 
 
 
 
 
 
 m 
 47662.31598042284 
 48229.69254328166 
 
 
 
 
 
 
 n 
 17303.053206656172 
 14762.900535368122 
 
 
 
 True 
 
 age40 
 m 
 -0.11943426225447411 
 0.05174539786330165 
 
 
 
 
 
 
 m 
 50360.35513714851 
 49183.82287650288 
 
 
 
 
 
 
 n 
 15114.99013027268 
 14423.26070433845 
 
 
 
 
 
 age60 
 m 
 -0.17052401128244693 
 0.1715553417547939 
 
 
 
 
 
 
 m 
 50358.62572299295 
 50243.500038446844 
 
 
 
 
 
 
 n 
 15356.269397920525 
 14194.466442890502

## gt_table with all nodes and custom formatting

Include non-analysis nodes and customize decimal places:

In [28]:
# Show all nodes with 2 decimal places
gt_table(
    res, 
    title="Complete Analysis Tree",
    cascade=True,
    decimals=2
).show()

Complete Analysis Tree 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 -- 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 Gender 
 -- 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 F 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49412.40832296735 
 
 
 
 
 
 
 
 
 n 
 15025.433301699664 
 
 
 
 
 
 
 
 
 m 
 0.07254044114026609 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 47662.31598042284 
 
 
 
 
 
 
 
 
 n 
 17303.053206656172 
 
 
 
 
 
 
 
 
 m 
 0.08039270194225605 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 50360.35513714851 
 
 
 
 
 
 
 
 
 n 
 15114.99013027268 
 
 
 
 
 
 
 
 
 m 
 -0.11943426225447411 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50358.62572299295 
 
 
 
 
 
 
 
 
 n 
 15356.269397920525 
 
 
 
 
 
 
 
 
 m 
 -0.17052401128244693 
 
 
 
 M 
 -- 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 Benin Y/N 
 -- 
 -- 
 -- 
 
 
 
 
 
 
 
 False 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 48211.790436672985 
 
 
 
 
 
 
 
 
 n 
 15381.549224855946 
 
 
 
 
 
 
 
 
 m 
 -0.13124823552832085 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 48229.69254328166 
 
 
 
 
 
 
 
 
 n 
 14762.900535368122 
 
 
 
 
 
 
 
 
 m 
 -0.0712858978586298 
 
 
 
 
 
 True 
 -- 
 -- 
 
 
 
 
 
 
 
 
 Age Group 
 -- 
 
 
 
 
 
 
 
 
 
 age40 
 
 
 
 
 
 
 
 
 
 
 m 
 49183.82287650288 
 
 
 
 
 
 
 
 
 n 
 14423.26070433845 
 
 
 
 
 
 
 
 
 m 
 0.05174539786330165 
 
 
 
 
 
 
 
 age60 
 
 
 
 
 
 
 
 
 
 
 m 
 50243.500038446844 
 
 
 
 
 
 
 
 
 n 
 14194.466442890502 
 
 
 
 
 
 
 
 
 m 
 0.1715553417547939

# Pivoting Statistics as Columns

The `pivot_statistics` parameter allows you to display statistics as columns instead of rows, creating a wider, more compact table format.

In [29]:
# Pivot statistics into columns - simple_table
simple_table(res, pivot_statistics=True)

statistics,_Level_0,_Level_1,_Level_2,_Level_3,_Level_4,_Level_5,m,n
0,Gender,F,Benin Y/N,False,Age Group,age40,0.07254,NaN
1,,,,,,,49412.408323,15025.433302
2,,,,,,age60,0.080393,NaN
3,,,,,,,47662.31598,17303.053207
4,,,,True,,age40,-0.119434,NaN
5,,,,,,,50360.355137,15114.99013
6,,,,,,age60,-0.170524,NaN
7,,,,,,,50358.625723,15356.269398
8,,M,,False,,age40,-0.131248,NaN
9,,,,,,,48211.790437,15381.549225


## Combining pivot_statistics with by

You can combine both pivots to create a matrix-style table with split groups and statistics as columns:

In [30]:
# Combine both pivots - shows Gender groups with statistics as columns
simple_table(res, by="df.Gender", pivot_statistics=True)

,_Level_0,_Level_1,_Level_2,_Level_3,F||m,F||n,M||m,M||n
0,Benin Y/N,False,Age Group,age40,0.07254,NaN,-0.131248,NaN
1,,,,,49412.408323,15025.433302,48211.790437,15381.549225
2,,,,age60,0.080393,NaN,-0.071286,NaN
3,,,,,47662.31598,17303.053207,48229.692543,14762.900535
4,,True,,age40,-0.119434,NaN,0.051745,NaN
5,,,,,50360.355137,15114.99013,49183.822877,14423.260704
6,,,,age60,-0.170524,NaN,0.171555,NaN
7,,,,,50358.625723,15356.269398,50243.500038,14194.466443


## gt_table with pivot_statistics

The same feature works with `gt_table()` for beautifully formatted output:

In [31]:
# gt_table with statistics as columns
gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    title="Analysis with Statistics as Columns"
).show()

Analysis with Statistics as Columns 
 
 
 
 Hierarchy 
 
 
 F 
 
 
 M 
 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 m 
 n 
 m 
 n 
 
 
 
 
 Benin Y/N 
 False 
 Age Group 
 age40 
 0.07254044114026609 
 
 -0.13124823552832085 
 
 
 
 
 
 
 
 49412.40832296735 
 15025.433301699664 
 48211.790436672985 
 15381.549224855946 
 
 
 
 
 
 age60 
 0.08039270194225605 
 
 -0.0712858978586298 
 
 
 
 
 
 
 
 47662.31598042284 
 17303.053206656172 
 48229.69254328166 
 14762.900535368122 
 
 
 
 True 
 
 age40 
 -0.11943426225447411 
 
 0.05174539786330165 
 
 
 
 
 
 
 
 50360.35513714851 
 15114.99013027268 
 49183.82287650288 
 14423.26070433845 
 
 
 
 
 
 age60 
 -0.17052401128244693 
 
 0.1715553417547939 
 
 
 
 
 
 
 
 50358.62572299295 
 15356.269397920525 
 50243.500038446844 
 14194.466442890502

In [ ]:
# gt_table combining both pivots
x = gt_table(
    res,
    by="df.Gender",
    pivot_statistics=True,
    cascade=True,
    title="Gender Comparison - Matrix View",
    subtitle="Each group-statistic combination as a column"
).show()

great_tables.gt.GT

In [33]:
from pyMyriad.tabular import format_statistics

# format the 'Income analysis' analysis (identified with its label) into a new statistic 'm' with a custom format, and remove the original statistics
rr = format_statistics(
    res, 
    label="Income analysis",  # Only format nodes with this label
    m="{m:.0f} /// {n:.0f}",         # Create statistic 'm' with this format
    inplace=False, 
    remove_original=True
)

gt_table(rr, title="Formatted Statistics").show()

Formatted Statistics 
 
 
 
 Hierarchy 
 
 Statistic 
 Value 
 
 
 _Level_0 
 _Level_1 
 _Level_2 
 _Level_3 
 _Level_4 
 _Level_5 
 
 
 
 
 Gender 
 F 
 Benin Y/N 
 False 
 Age Group 
 age40 
 m 
 49412 /// 15025 
 
 
 
 
 
 
 
 
 m 
 0.07254044114026609 
 
 
 
 
 
 
 
 age60 
 m 
 47662 /// 17303 
 
 
 
 
 
 
 
 
 m 
 0.08039270194225605 
 
 
 
 
 
 True 
 
 age40 
 m 
 50360 /// 15115 
 
 
 
 
 
 
 
 
 m 
 -0.11943426225447411 
 
 
 
 
 
 
 
 age60 
 m 
 50359 /// 15356 
 
 
 
 
 
 
 
 
 m 
 -0.17052401128244693 
 
 
 
 M 
 
 False 
 
 age40 
 m 
 48212 /// 15382 
 
 
 
 
 
 
 
 
 m 
 -0.13124823552832085 
 
 
 
 
 
 
 
 age60 
 m 
 48230 /// 14763 
 
 
 
 
 
 
 
 
 m 
 -0.0712858978586298 
 
 
 
 
 
 True 
 
 age40 
 m 
 49184 /// 14423 
 
 
 
 
 
 
 
 
 m 
 0.05174539786330165 
 
 
 
 
 
 
 
 age60 
 m 
 50244 /// 14194 
 
 
 
 
 
 
 
 
 m 
 0.1715553417547939